# DPSynth Quickstart Demo

This notebook begins with the lightweight In-Memory DataFrame API, which is
recommended for getting started with DPSynth.

An optional second section demonstrates the Scalable Beam API. Its additional
dependencies are installed only when that section is reached.

In [ ]:
# @title Install DPSynth
!pip install "dpsynth @ git+https://github.com/google/dpsynth.git"

The base installation is sufficient for the in-memory API. It avoids installing the optional Apache Beam, TensorFlow, text-generation, and example-evaluation dependencies.

## In-memory DataFrame API

## Download Dataset

To run this notebook with a realistic dataset, we will use the Adult Income dataset from Kaggle. You will need a Kaggle account and an API token (`kaggle.json`) to download it.

In [ ]:
# Install Kaggle API
!pip install kagglehub

import kagglehub

kagglehub.login()

# If you are using Colab, you can alternatively set KAGGLE_USERNAME and KAGGLE_KEY
# values in user data, and then uncomment and run the following code:
#
# from colabtools import userdata
#
# os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
# os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
#
# You use userdata to keep the Kaggle API key safe. Alternatively, you can
# hardcode the values but it is not recommended due to security risks of
# leaking the API key.


# If you're not using Colab, set the env vars as appropriate for your system.
# For example, to set the env vars on Linux you can run in terminal:
# ```
# export KAGGLE_USERNAME="your_username"
# export KAGGLE_KEY="your_key"
# ```

# Download latest version
ds_path = kagglehub.dataset_download("wenruliu/adult-income-dataset")

print("Path to dataset files:", ds_path)

In [ ]:
import os
import dpsynth
from dpsynth import discrete_mechanisms
from dpsynth import domain
import numpy as np
import pandas as pd

# 1. Load the downloaded dataset
dataset_path = os.path.join(ds_path, 'adult.csv')
sensitive_df = pd.read_csv(dataset_path)

print('Original Sensitive Data (sample):')
print(sensitive_df[['age', 'education']].head())

# 2. Define domain schema
# We consider age as Categorical to avoid discretization issues as suggested by reviewer.
attribute_domains = {
    'age': domain.CategoricalAttribute(
        possible_values=list(range(17, 91))
    ),  # Adult dataset age range
    'education': domain.CategoricalAttribute(
        possible_values=sensitive_df['education'].unique().tolist()
    ),
}

# 3. Configure and calibrate the synthesis mechanism (MST as default)
config = dpsynth.TabularConfig(
    domains=attribute_domains,
    discrete_mechanism=discrete_mechanisms.MSTConfig(),
)
mechanism = config.calibrate(epsilon=1.0, delta=1e-5)

# 4. Generate Differentially Private synthetic data
result = mechanism(
    np.random.default_rng(),
    sensitive_df[
        ['age', 'education']
    ],  # Restrict to these columns for simplicity
)
synthetic_df = result.synthetic_data

print('\nGenerated Synthetic Data (sample):')
print(synthetic_df.head())

## Optional: Scalable Beam API Example

The following section requires the pipeline dependencies, including Apache Beam
and TensorFlow. You can skip this section if you only need the in-memory API.

In [ ]:
# @title Install optional pipeline dependencies
!pip install "dpsynth[pipeline] @ git+https://github.com/google/dpsynth.git"

In [ ]:
import apache_beam as beam
from dpsynth import data_generation
from dpsynth.dataset_descriptors import csv_descriptor
from dpsynth.pipeline_transformations import input_output
from dpsynth.pipeline_transformations import types
import pandas as pd
import pipeline_dp

# 1. Create a descriptor for the selected CSV columns. Categorical domains that
# are not provided publicly are derived privately during pipeline execution.

sample_df = pd.read_csv(dataset_path, nrows=100)
descriptor = csv_descriptor.get_dataset_descriptor_for_csv(
    dataframe=sample_df, field_names=['age', 'education']
)


# 2. Configure the synthesis mechanism
config = data_generation.DataGenerationConfig(
    epsilon=1.0,
    delta=1e-5,
    mechanism=data_generation.Mechanism.MST,
    dataset_descriptor=descriptor,
    data_format=types.DataFormat.CSV,
    num_out_records=1000,
)

# 3. Run the pipeline with BeamBackend
with beam.Pipeline() as pipeline:
  # Load data from CSV file in Beam

  raw_records = input_output.load_data_for_beam(
      pipeline,
      dateset_path,
      types.DataFormat.CSV,
  )

  beam_backend = pipeline_dp.BeamBackend()

  synthetic_records = data_generation.generate(
      input_data=raw_records,
      config=config,
      backend=beam_backend,
  )

  # Print sample of results
  synthetic_records | 'Sample' >> beam.combiners.Sample.FixedSizeGlobally(
      5
  ) | 'Print' >> beam.Map(print)